## pyspark.sql.functions.groupBy()

groupBy(): 

Similar to SQL GROUP BY clause, PySpark groupBy() transformation that is used to group rows that have the same values in specified columns into summary rows. It allows you to perform aggregate functions on groups of rows, rather than on individual rows, enabling you to summarize data and generate aggregate statistics.

References: 
- doc: https://sparkbyexamples.com/pyspark/pyspark-groupby-explained-with-example/
- code: https://github.com/spark-examples/pyspark-examples/blob/master/pyspark-groupby.py


### GroupBy() Syntax & Simple Usage

Syntax:
- DataFrame.groupBy(*cols) OR
- DataFrame.groupby(*cols)
- NOTE: *cols: allow group by multiple columns.  For example: df.groupBy("department","state")
  
Usage: 
- When we perform groupBy() on PySpark Dataframe, it returns GroupedData object which contains below aggregate functions.

|Function	|Definition|
|--------   |----------|
|count()	|Use groupBy() count() to return the number of rows for each group.|
|mean()	    |Returns the mean of values for each group.|
|max()	    |Returns the maximum of values for each group.|
|min()	    |Returns the minimum of values for each group.|
|sum()	    |Returns the total for values for each group.|
|avg()	    |Returns the average for values for each group.|
|agg()	    |Using groupBy() agg() function, we can calculate more than one aggregate at a time.|
|pivot()	|This function is used to Pivot the DataFrame, which I will not cover in this article as I already have a dedicated article for Pivot & Unpivot DataFrame.|


In [7]:
# Prepare data

from pyspark.sql import SparkSession

# Initialize SparkSession
spark = SparkSession.builder \
    .appName("example") \
    .getOrCreate()

# Data
simpleData = [("James","Sales","NY",90000,34,10000),
    ("Michael","Sales","NY",86000,56,20000),
    ("Robert","Sales","CA",81000,30,23000),
    ("Maria","Finance","CA",90000,24,23000),
    ("Raman","Finance","CA",99000,40,24000),
    ("Scott","Finance","NY",83000,36,19000),
    ("Jen","Finance","NY",79000,53,15000),
    ("Jeff","Marketing","CA",80000,25,18000),
    ("Kumar","Marketing","NY",91000,50,21000)
  ]

# Create DataFrame
schema = ["employee_name","department","state","salary","age","bonus"]
df = spark.createDataFrame(data=simpleData, schema = schema)
df.printSchema()
df.show(truncate=False)

root
 |-- employee_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- state: string (nullable = true)
 |-- salary: long (nullable = true)
 |-- age: long (nullable = true)
 |-- bonus: long (nullable = true)

+-------------+----------+-----+------+---+-----+
|employee_name|department|state|salary|age|bonus|
+-------------+----------+-----+------+---+-----+
|James        |Sales     |NY   |90000 |34 |10000|
|Michael      |Sales     |NY   |86000 |56 |20000|
|Robert       |Sales     |CA   |81000 |30 |23000|
|Maria        |Finance   |CA   |90000 |24 |23000|
|Raman        |Finance   |CA   |99000 |40 |24000|
|Scott        |Finance   |NY   |83000 |36 |19000|
|Jen          |Finance   |NY   |79000 |53 |15000|
|Jeff         |Marketing |CA   |80000 |25 |18000|
|Kumar        |Marketing |NY   |91000 |50 |21000|
+-------------+----------+-----+------+---+-----+



In [16]:
# groupBy().count()
df.groupBy("department").count().show()

# groupBy().avg()
df.groupBy("department").avg( "salary").show()

# groupBy().min()
df.groupBy("department").min("salary").show()

# groupBy().max()
df.groupBy("department").max("salary").show()

# groupBy().mean()
df.groupBy("department").mean( "salary").show()

# groupBy().sum()
df.groupBy("department").sum("salary").show()


+----------+-----+
|department|count|
+----------+-----+
|     Sales|    3|
|   Finance|    4|
| Marketing|    2|
+----------+-----+

+----------+-----------------+
|department|      avg(salary)|
+----------+-----------------+
|     Sales|85666.66666666667|
|   Finance|          87750.0|
| Marketing|          85500.0|
+----------+-----------------+

+----------+-----------+
|department|min(salary)|
+----------+-----------+
|     Sales|      81000|
|   Finance|      79000|
| Marketing|      80000|
+----------+-----------+

+----------+-----------+
|department|max(salary)|
+----------+-----------+
|     Sales|      90000|
|   Finance|      99000|
| Marketing|      91000|
+----------+-----------+

+----------+-----------------+
|department|      avg(salary)|
+----------+-----------------+
|     Sales|85666.66666666667|
|   Finance|          87750.0|
| Marketing|          85500.0|
+----------+-----------------+

+----------+-----------+
|department|sum(salary)|
+----------+-----------+
|  

### GroupBy() Advance Usage

In [12]:
# groupBy on multiple columns and calculate sum on multiple columns
df.groupBy("department","state") \
    .sum("salary","bonus") \
    .show()

# Result:
#+----------+-----+-----------+----------+
#|department|state|sum(salary)|sum(bonus)|
#+----------+-----+-----------+----------+
#|     Sales|   NY|     176000|     30000|
#|     Sales|   CA|      81000|     23000|
#|   Finance|   CA|     189000|     47000|
#|   Finance|   NY|     162000|     34000|
#| Marketing|   CA|      80000|     18000|
#| Marketing|   NY|      91000|     21000|
#+----------+-----+-----------+----------+


+----------+-----+-----------+----------+
|department|state|sum(salary)|sum(bonus)|
+----------+-----+-----------+----------+
|     Sales|   NY|     176000|     30000|
|     Sales|   CA|      81000|     23000|
|   Finance|   CA|     189000|     47000|
|   Finance|   NY|     162000|     34000|
| Marketing|   CA|      80000|     18000|
| Marketing|   NY|      91000|     21000|
+----------+-----+-----------+----------+



In [17]:
# groupBy with mutiple aggregations using agg function

from pyspark.sql.functions import sum, avg, max

df.groupBy("department") \
    .agg(sum("salary").alias("sum_salary"), \
         avg("salary").alias("avg_salary"), \
         sum("bonus").alias("sum_bonus"), \
         max("bonus").alias("max_bonus") \
     ) \
    .show()

+----------+----------+-----------------+---------+---------+
|department|sum_salary|       avg_salary|sum_bonus|max_bonus|
+----------+----------+-----------------+---------+---------+
|     Sales|    257000|85666.66666666667|    53000|    23000|
|   Finance|    351000|          87750.0|    81000|    24000|
| Marketing|    171000|          85500.0|    39000|    21000|
+----------+----------+-----------------+---------+---------+



In [21]:
# Using filter on aggregate data using where

from pyspark.sql.functions import col, sum, avg, max

df.groupBy("department") \
    .agg(sum("salary").alias("sum_salary"), \
      avg("salary").alias("avg_salary"), \
      sum("bonus").alias("sum_bonus"), \
      max("bonus").alias("max_bonus")) \
    .where(col("sum_bonus") >= 50000) \
    .show(truncate=False)

+----------+----------+-----------------+---------+---------+
|department|sum_salary|avg_salary       |sum_bonus|max_bonus|
+----------+----------+-----------------+---------+---------+
|Sales     |257000    |85666.66666666667|53000    |23000    |
|Finance   |351000    |87750.0          |81000    |24000    |
+----------+----------+-----------------+---------+---------+



### Spark SQL group by query instead of groupBy function 

In [22]:
# Spark SQL query for results similar to previous example
df.createOrReplaceTempView("employees")

# Using SQL Query
sql_string = """SELECT department,
       SUM(salary) AS sum_salary,
       AVG(salary) AS avg_salary,
       SUM(bonus) AS sum_bonus,
       MAX(bonus) AS max_bonus
FROM employees
GROUP BY department
HAVING SUM(bonus) >= 50000"""

# Execute SQL query against the temporary view
df2 = spark.sql(sql_string)
df2.show()

+----------+----------+-----------------+---------+---------+
|department|sum_salary|       avg_salary|sum_bonus|max_bonus|
+----------+----------+-----------------+---------+---------+
|     Sales|    257000|85666.66666666667|    53000|    23000|
|   Finance|    351000|          87750.0|    81000|    24000|
+----------+----------+-----------------+---------+---------+

